In [1]:
import sys
sys.path.append(".")
sys.path.append("./collisionChecker")

import matplotlib.pyplot as plt
import time
import importlib
%matplotlib inline

from shapely.geometry import Point, LineString
from shapely import plotting

# Benchmark Suite importieren (mit Reload falls schon geladen)
import IPTestSuiteBenchmark
import IPTestSuitePlanarBenchmark
importlib.reload(IPTestSuiteBenchmark)
importlib.reload(IPTestSuitePlanarBenchmark)
from IPTestSuiteBenchmark import benchList, getBenchmarks2DoF, getBenchmarks3DoF, printBenchmarkOverview

# Planer importieren
from IPLazyPRM import LazyPRM
from IPBasicPRM import BasicPRM
from IPVisibilityPRM import VisPRM
from IPVisibilityPRMRound import VisPRMRound
from IPRoundtripPlanner import RoundtripPlanner

#Animator importieren
from PathAnimator import PathAnimator
from shapely import plotting


# Übersicht
printBenchmarkOverview()
print(f"\nAnzahl Benchmarks geladen: {len(benchList)}")
for i, b in enumerate(benchList):
    print(f"  {i+1}. {b.name}")

BENCHMARK ÜBERSICHT
1. doubleBowl_2DoF (2-DoF, Level 1)
   Start: [11, 9]
   Goals: [[17, 17], [3, 3], [25, 5], [15, 27]]
2. arcade_2DoF (2-DoF, Level 2)
   Start: [25, 5]
   Goals: [[5, 20], [25, 27], [5, 10], [10, 5]]
3. spinner_3DoF (3-DoF, Level 2)
   Start: [15, 5, 0]
   Goals: [[5, 15, 270], [15, 25, 180], [25, 15, 90]]
4. grid_3DoF (3-DoF, Level 3)
   Start: [2.5, 2.5, 0]
   Goals: [[27, 7, 180], [17, 28, 90], [7, 17, 270]]

Anzahl Benchmarks geladen: 4
  1. doubleBowl_2DoF
  2. arcade_2DoF
  3. spinner_3DoF
  4. grid_3DoF


In [ ]:
PLANNERS = {
    "LazyPRM": {
        "config": {
            "initialRoadmapSize": 40,
            "updateRoadmapSize": 20,
            "kNearest": 5,
            "maxIterations": 40
        },
        "class": LazyPRM
    },
    "BasicPRM": {
        "config": {
            "radius": 5.0,             # Suchradius für Nachbarn
            "numNodes": 300,           # Anzahl der zu generierenden Knoten
            "useKDTree": True          # KDTree für schnelle Nachbarsuche (optional)
        },
        "class": BasicPRM
    },
    "VisPRM": {
        "config": {
            "ntry": 40                 # Anzahl Versuche für Roadmap-Erstellung
        },
        "class": VisPRM,
    },
    "VisPRMRound": {
        "config": {
            "ntry": 40                # Roundtrip-Variante nutzt eigene TSP-Logik
        },
        "class": VisPRMRound
    },
}

In [ ]:
from IPTestSuitePlanarBenchmark import benchList
import IPPlanarManipulator
benchmarks_planar = [b for b in benchList if b.name in ["PlanarArm_2DoF", "PlanarArm_3DoF"]]
planner_names = list(PLANNERS.keys())
results_planar = []

for bench in benchmarks_planar:
    print(f"\nBenchmark: {bench.name}")
    print("-" * 50)
    env = bench.collisionChecker
    start_pos = bench.startList[0]
    goal_list = bench.goalList

    for planner_name in planner_names:
        try:
            planner_class = PLANNERS[planner_name]["class"]
            planner_config = PLANNERS[planner_name]["config"]
            base_planner = planner_class(env)

            before = getattr(env, "collision_calls", 0)
            start_time = time.time()
            if planner_name == "VisPRMRound":
                path, length = base_planner.planPath([start_pos], goal_list, planner_config)
            else:
                roundtrip = RoundtripPlanner(env, base_planner)
                path,length = roundtrip.planPath([start_pos], goal_list, planner_config)
            end_time = time.time()

            after = getattr(env, "collision_calls", 0)
            n_collisions = after - before

            roadmap_size = getattr(base_planner, "graph", None)
            if roadmap_size is not None and hasattr(roadmap_size, "size"):
                roadmap_size = roadmap_size.size()
            else:
                roadmap_size = None

            planning_time = end_time - start_time
            path_length = len(path)

            results_planar.append({
                "benchmark": bench.name,
                "planner": planner_name,
                "time": planning_time,
                "path_points": path_length,
                "path": path,
                "success": True,
                "collision_checks": n_collisions,
                "roadmap_size": roadmap_size,
                "tsp_total_dist": length,
            })
            print(f"  {planner_name}: {planning_time:.3f}s, {path_length} Punkte, {n_collisions} CollisionChecks, Roadmap: {roadmap_size}, TSP Dist: {length:.2f}")

        except Exception as e:
            results_planar.append({
                "benchmark": bench.name,
                "planner": planner_name,
                "time": None,
                "path_points": None,
                "path": None,
                "success": False,
                "collision_checks": None,
                "roadmap_size": None,
                "length": None,
            })
            print(f"  {planner_name}: Fehler - {str(e)[:50]}...")

print("\n" + "=" * 70)

In [ ]:
# ============================================================================
# ERGEBNISSE PLOTTEN: Planar Manipulator (Arm)
# ============================================================================
import numpy as np

# Nur erfolgreiche Ergebnisse filtern
successful_results_planar = [r for r in results_planar if r['success']]

# Anzahl Plots bestimmen
n_benchmarks_planar = len(benchmarks_planar)
n_planners_planar = len(planner_names)

if successful_results_planar and n_benchmarks_planar > 0:
    fig, axes = plt.subplots(n_benchmarks_planar, n_planners_planar,
                             figsize=(5*n_planners_planar, 5*n_benchmarks_planar))

    # Sicherstellen, dass axes immer ein 2D-Array ist (auch bei 1x1 oder 1xN Plots)
    if n_benchmarks_planar == 1 and n_planners_planar == 1:
        axes = np.array([[axes]])
    elif n_benchmarks_planar == 1:
        axes = np.array([axes])
    elif n_planners_planar == 1:
        axes = np.array([axes]).reshape(-1, 1)

    for bench_idx, bench in enumerate(benchmarks_planar):
        for plan_idx, planner_name in enumerate(planner_names):
            ax = axes[bench_idx, plan_idx]

            # Das passende Ergebnis für diesen Benchmark und Planer suchen
            result = next((r for r in successful_results_planar
                           if r['benchmark'] == bench.name and r['planner'] == planner_name), None)

            env = bench.collisionChecker

            # --- 1. Arbeitsraum-Limits setzen ---
            # WICHTIG: env.getEnvironmentLimits() liefert bei KinChain oft die Winkel-Limits (z.B. -3.14 bis 3.14).
            # Wir brauchen aber die Koordinaten für den Plot (z.B. 0 bis 30).
            # Da diese nicht direkt gespeichert sind, setzen wir hier feste Werte passend zur Szene (ca. 30x30).
            ax.set_xlim(0, 30)
            ax.set_ylim(0, 30)
            ax.grid(True, alpha=0.3)
            ax.set_aspect('equal')

            # --- 2. Hindernisse zeichnen ---
            # Bei KinChainCollisionChecker muss explizit inWorkspace=True gesetzt werden!
            env.drawObstacles(ax, inWorkspace=True)

            title = f"{bench.name}\n{planner_name}"

            if result and result['path']:
                path = result['path']
                title += f": {result['time']:.2f}s, {len(path)} Pkt."

                # Hilfsfunktion, um den Roboter in einer bestimmten Farbe zu zeichnen
                def plot_robot_chain(environment, configuration, ax_obj, color, alpha=1.0, linestyle='-'):
                    environment.kin_chain.move(configuration)
                    transforms = environment.kin_chain.get_transforms()
                    # Segmente zeichnen
                    xs = [t[0] for t in transforms]
                    ys = [t[1] for t in transforms]
                    ax_obj.plot(xs, ys, color=color, alpha=alpha, linewidth=2, linestyle=linestyle)
                    # Gelenke als Punkte
                    ax_obj.plot(xs, ys, 'o', color=color, alpha=alpha, markersize=4)

                # --- 3. Pfad visualisieren (Stroboskop-Effekt) ---
                # Wir zeichnen den Roboter an ca. 10-15 Zwischenpositionen
                step = max(1, len(path) // 15)

                # Endeffektor-Spur speichern
                ee_trace_x = []
                ee_trace_y = []

                for i in range(0, len(path), step):
                    # Zwischenschritte transparent blau
                    plot_robot_chain(env, path[i], ax, color='blue', alpha=0.15)

                    # Endeffektor Position für Spur speichern
                    current_eff = env.kin_chain.get_transforms()[-1]
                    ee_trace_x.append(current_eff[0])
                    ee_trace_y.append(current_eff[1])

                # Letzten Punkt auch hinzufügen für Spur
                env.kin_chain.move(path[-1])
                last_eff = env.kin_chain.get_transforms()[-1]
                ee_trace_x.append(last_eff[0])
                ee_trace_y.append(last_eff[1])

                # Spur des Endeffektors zeichnen (gestrichelt)
                ax.plot(ee_trace_x, ee_trace_y, 'b--', linewidth=1, alpha=0.6, label='TCP Spur')

                # --- 4. Start und Ziele hervorheben ---

                # Start (Grün, voll deckend)
                plot_robot_chain(env, path[0], ax, color='green', alpha=1.0)

                # Alle Ziele aus der Benchmark-Definition (Rot)
                for goal_conf in bench.goalList:
                    plot_robot_chain(env, goal_conf, ax, color='red', alpha=0.6, linestyle='--')

            else:
                title += ": FEHLER / Kein Pfad"

            ax.set_title(title, fontsize=10)

    plt.tight_layout()
    plt.show()

    # Tabelle für Planar ausgeben
    print("\nERGEBNISTABELLE  ")
    print("-" * 100)
    print(f"{'Benchmark':<20} {'Planner':<15} {'Zeit (s)':<12} {'Punkte':<10} {'Roadmap':<10} {'CollChecks':<12} {'TSP Dist':<12}")
    print("-" * 100)
    for r in successful_results_planar:
        tsp_dist = r.get('tsp_total_dist', None)
        tsp_str = f"{tsp_dist:.2f}" if tsp_dist is not None else "-"

        print(f"{r['benchmark']:<20} {r['planner']:<15} {r['time']:<12.3f} {r['path_points']:<10} {r['roadmap_size']:<10} {r['collision_checks']:<12} {tsp_str:<12}")

else:
    print("Keine erfolgreichen Planar-Ergebnisse zum Plotten vorhanden.")

In [ ]:
# ============================================================================
# INLINE-ANIMATION: Planar Roboter (Anzeige direkt im Notebook)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Pfad-Interpolation (wie zuvor, damit Bewegungen weich sind)
def interpolate_path_robust(path, num_frames=50):
    if not path or len(path) < 2:
        return np.array(path if path else [])
    path = np.array(path)
    dists = np.linalg.norm(np.diff(path, axis=0), axis=1)
    total_dist = np.sum(dists)
    if total_dist == 0: return np.array([path[0]] * num_frames)
    cum_dist = np.r_[0, np.cumsum(dists)]
    target_dists = np.linspace(0, total_dist, num_frames)
    interpolated = np.zeros((num_frames, path.shape[1]))
    for i in range(path.shape[1]):
        interpolated[:, i] = np.interp(target_dists, cum_dist, path[:, i])
    return interpolated

# Filtern der erfolgreichen Ergebnisse
successful_planar = [r for r in results_planar if r['success']]

print(f"Generiere {len(successful_planar)} Animationen für das Notebook...\n(Das kann einen Moment dauern)")

for res in successful_planar:
    bench_name = res['benchmark']
    planner_name = res['planner']
    path = res['path']

    # Benchmark wiederfinden
    bench = next((b for b in benchmarks_planar if b.name == bench_name), None)
    if not bench: continue

    env = bench.collisionChecker

    # --- 1. Plot Setup ---
    # Wir erstellen den Plot, aber zeigen ihn noch nicht an (plt.ioff könnte man nutzen, aber close reicht)
    fig, ax = plt.subplots(figsize=(6, 6))

    # KORREKTE LIMITS für den Arbeitsraum setzen (0 bis 35)
    # Sonst sieht man nur einen winzigen Ausschnitt oder nichts
    ax.set_xlim(0, 35)
    ax.set_ylim(0, 35)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f"{bench_name} : {planner_name}\n(Zeit: {res['time']:.2f}s, Punkte: {len(path)})", fontsize=10)

    # --- 2. Hindernisse zeichnen ---
    # Wir nutzen shapely direkt, um sicher zu sein
    if hasattr(env, 'scene'):
        from shapely import plotting
        for val in env.scene.values():
            plotting.plot_polygon(val, ax=ax, add_points=False, color='red', alpha=0.5)

    # --- 3. Animationselemente ---
    # Pfad interpolieren (ca. 40-50 Frames für Performance im Notebook)
    anim_path = interpolate_path_robust(path, num_frames=45)

    # Roboterarm (Blau)
    line_arm, = ax.plot([], [], 'o-', linewidth=3, markersize=6, color='blue', label='Roboter')
    # Spur (Orange)
    trace_x, trace_y = [], []
    line_trace, = ax.plot([], [], '--', linewidth=1, color='orange', alpha=0.6, label='TCP Spur')

    # Start (Grün, statisch)
    env.kin_chain.move(bench.startList[0])
    ts = env.kin_chain.get_transforms()
    ax.plot([t[0] for t in ts], [t[1] for t in ts], 'o-', color='green', alpha=0.4, linewidth=2)

    # Ziele (Rot, statisch)
    for goal in bench.goalList:
        env.kin_chain.move(goal)
        ts = env.kin_chain.get_transforms()
        ax.plot([t[0] for t in ts], [t[1] for t in ts], 'o-', color='red', alpha=0.4, linewidth=2)

    ax.legend(loc='upper right', fontsize=8)

    # --- 4. Update-Funktion ---
    def update(frame):
        # Roboter bewegen
        config = anim_path[frame]
        env.kin_chain.move(config)

        # Koordinaten auslesen
        transforms = env.kin_chain.get_transforms()
        xs = [t[0] for t in transforms]
        ys = [t[1] for t in transforms]

        # Linien aktualisieren
        line_arm.set_data(xs, ys)

        trace_x.append(xs[-1])
        trace_y.append(ys[-1])
        line_trace.set_data(trace_x, trace_y)
        return line_arm, line_trace

    # --- 5. Rendern & Anzeigen ---
    try:
        ani = FuncAnimation(fig, update, frames=len(anim_path), interval=60, blit=True)
        # Rendert die Animation als HTML5/JS-Widget
        display(HTML(ani.to_jshtml()))
    except Exception as e:
        print(f"Fehler bei {bench_name}/{planner_name}: {e}")

    # Wichtig: Figure schließen, damit sie nicht doppelt als statisches Bild erscheint
    plt.close(fig)

print("Alle Animationen gerendert.")

In [ ]:
# ============================================================================
# ERGEBNISSE ALS BALKENDIAGRAMME - PLANAR
# ============================================================================
import numpy as np

successful_results_planar = [r for r in results_planar if r['success']]

if successful_results_planar:
    fig, axes = plt.subplots(3, 2, figsize=(12, 8))  # 2x2 Grid

    # Daten vorbereiten
    benchmarks_planar_names = list(set(r['benchmark'] for r in successful_results_planar))
    x = np.arange(len(benchmarks_planar_names))
    width = 0.18

    # Plot 1: Planungszeit (oben links)
    ax1 = axes[0, 0]
    for i, planner in enumerate(planner_names):
        times = [next((r['time'] for r in successful_results_planar
                      if r['benchmark'] == b and r['planner'] == planner), 0)
                for b in benchmarks_planar_names]
        ax1.bar(x + i*width, times, width, label=planner, alpha=0.8)
    ax1.set_ylabel('Planungszeit (s)')
    ax1.set_title('Planungszeit pro Benchmark')
    ax1.set_xticks(x + width)
    ax1.set_xticklabels(benchmarks_planar_names, rotation=15)
    ax1.legend(framealpha=0.5)
    ax1.grid(True, alpha=0.3)

    # Plot 2: Pfadpunkte pro Planer
    ax2 = axes[0, 1]
    for i, planner in enumerate(planner_names):
        points = [next((r['path_points'] for r in successful_results_planar
                       if r['benchmark'] == b and r['planner'] == planner), 0)
                 for b in benchmarks_planar_names]
        ax2.bar(x + i*width, points, width, label=planner, alpha=0.8)
    ax2.set_ylabel('Pfadpunkte')
    ax2.set_title('Pfadpunkte pro Benchmark')
    ax2.set_xticks(x + width)
    ax2.set_xticklabels(benchmarks_planar_names, rotation=15)
    ax2.legend(framealpha=0.5)
    ax2.grid(True, alpha=0.3)

    # Plot 3: Roadmap Size (unten links)
    ax3 = axes[1, 0]
    for i, planner in enumerate(planner_names):
        roadmap_sizes = [next((r['roadmap_size'] for r in successful_results_planar
                              if r['benchmark'] == b and r['planner'] == planner), 0)
                        for b in benchmarks_planar_names]
        ax3.bar(x + i*width, roadmap_sizes, width, label=planner, alpha=0.8)
    ax3.set_ylabel('Roadmap Size')
    ax3.set_title('Roadmap Size pro Benchmark')
    ax3.legend(framealpha=0.5)

    ax4 = axes[1, 1]
    for i, planner in enumerate(planner_names):
        coll_checks = [next((r['collision_checks'] for r in successful_results_planar
                            if r['benchmark'] == b and r['planner'] == planner), 0)
                        for b in benchmarks_planar_names]
        ax4.bar(x + i*width, coll_checks, width, label=planner, alpha=0.8)
    ax4.set_ylabel('Collision Checks')
    ax4.set_title('Collision Checks pro Benchmark')
    ax4.set_xticks(x + width)
    ax4.set_xticklabels(benchmarks_planar_names, rotation=15)
    ax4.legend(framealpha=0.5)
    ax4.grid(True, alpha=0.3)

    # Plot 4: Pfadlaegne unten links
    ax5 = axes[2, 0]
    for i, planner in enumerate(planner_names):
        tsp_dists = [next((r.get('tsp_total_dist', 0) for r in successful_results_planar
                          if r['benchmark'] == b and r['planner'] == planner), 0)
                    for b in benchmarks_planar_names]
        ax5.bar(x + i*width, tsp_dists, width, label=planner, alpha=0.8)
    ax5.set_ylabel('Distanz in m')
    ax5.set_title('Pfadlaenge (Distanz in m) pro Benchmark')
    ax5.set_xticks(x + width)
    ax5.set_xticklabels(benchmarks_planar_names, rotation=15)
    ax5.legend(framealpha=0.5)
    ax5.grid(True, alpha=0.3)

    axes[2, 1].axis('off')  # Letztes Feld leer lassen

    plt.subplots_adjust(hspace=0.5, wspace=0.3)